# Compare two simulation runs

Point this at two results directories and it reports, measure by measure, whether they
agree — and when they do not, whether the difference is bigger than the run's own
seed-to-seed noise.

**Intended use:** a previous iteration vs. the current one. The comparison helpers are
defined once, near the top; the cells after them are thin calls. Add, remove, or reorder
those freely — that is the point of the notebook.

**A difference is not automatically a bug.** Any change to the RNG stream (a vivarium
version bump, a component reordering) moves every per-simulant outcome, so runs that are
scientifically equivalent will still disagree seed by seed. What matters is whether the
*distribution* moved. That is what the verdicts below measure.

## Parameters

Tagged `parameters` for papermill, matching the other notebooks in this directory
(`papermill compare_runs.ipynb out.ipynb -p RUN_A ... -p RUN_B ...`).

Each path may point at any of:

* a directory of flat `<measure>.parquet` files (older psimulate output),
* a directory of `<measure>/<hash>.parquet` shards (newer psimulate output),
* a psimulate `-o` directory — the newest `<timestamp>/results/` below it is used.

In [ ]:
# Run A -- the reference / previous iteration.
RUN_A = "/mnt/team/simulation_science/pub/models/vivarium_gates_lsff_by_wealth_quintile/2026_08_13_13_55_48/0200_pregnancy_sim/sim_results/rice/nigeria"
LABEL_A = "old (2026_08_13)"

# Run B -- the new / current iteration.
RUN_B = "/mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model1.0/maternal/rice/nigeria"
LABEL_B = "new (model1.0)"

# Measures renamed between the two runs: {name_in_A: name_in_B}.
MEASURE_ALIASES = {"person_time": "person_time_population"}

# Column holding the intervention scenario. Maternal uses "scenario"; the child sim uses
# "maternal_scenario". None -> auto-detect.
SCENARIO_COL = None

# Restrict the headline comparison to one scenario so a scenario mix cannot mask a shift.
# None -> compare the total across all scenarios.
SCENARIO = "baseline"

# |Welch t| on per-seed totals above which a measure is flagged.
T_REVIEW = 2.0
T_DIFFERS = 3.0

## Setup

In [ ]:
from __future__ import annotations

import glob
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from scipy import stats as _scipy_stats
except ImportError:
    _scipy_stats = None

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", message=".*observed=False.*")

# Categorical slots 1 and 2 of the validated default palette. Two series only, assigned in
# fixed order -- A is always blue, B always orange, regardless of which measure is plotted.
COLOR_A = "#2a78d6"
COLOR_B = "#eb6834"
# Status palette (fixed, never reused as a series color). Always paired with a text label.
STATUS_COLORS = {
    "IDENTICAL": "#0ca30c",
    "OK": "#0ca30c",
    "REVIEW": "#fab219",
    "DIFFERS": "#d03b3b",
    "N/A": "#8a8880",
}
INK = "#0b0b0b"
INK_MUTED = "#52514e"
SURFACE = "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": "#d8d6d0",
    "axes.labelcolor": INK_MUTED,
    "axes.titlecolor": INK,
    "axes.titlesize": 11,
    "axes.titleweight": "semibold",
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.color": "#e8e6e0",
    "grid.linewidth": 0.8,
    "text.color": INK,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "legend.frameon": False,
    "font.size": 10,
    "figure.dpi": 110,
})

### Loading

`resolve_results_dir` absorbs the layout differences so the rest of the notebook does not
care which psimulate version produced a run. `load_measure` returns a single tidy frame
either way, because `pd.read_parquet` concatenates a directory of shards.

In [ ]:
def resolve_results_dir(path: str | Path) -> Path:
    """Return the directory that directly contains the measures.

    Accepts a flat results dir, a sharded results dir, or a psimulate ``-o`` directory
    (in which case the newest ``<timestamp>/results`` beneath it wins).
    """
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(p)
    if _measure_names(p):
        return p
    nested = sorted(p.glob("*/results"))
    if nested:
        return nested[-1]
    nested = sorted(p.glob("*/*/results"))
    if nested:
        return nested[-1]
    raise FileNotFoundError(f"no measures and no */results under {p}")


def _measure_names(d: Path) -> set[str]:
    """Measure names directly under ``d``, from either layout."""
    flat = {f.stem for f in d.glob("*.parquet")}
    sharded = {sub.name for sub in d.iterdir() if sub.is_dir() and any(sub.glob("*.parquet"))}
    return flat | sharded


def list_measures(run: str | Path) -> set[str]:
    return _measure_names(resolve_results_dir(run))


def load_measure(run: str | Path, measure: str) -> pd.DataFrame:
    """Load one measure, whichever layout the run uses."""
    d = resolve_results_dir(run)
    flat, sharded = d / f"{measure}.parquet", d / measure
    if flat.exists():
        return pd.read_parquet(flat)
    if sharded.is_dir():
        return pd.read_parquet(sharded)
    raise FileNotFoundError(f"{measure} not found in {d}")


def describe_run(run: str | Path, label: str) -> pd.Series:
    d = resolve_results_dir(run)
    measures = sorted(_measure_names(d))
    layout = "flat" if (d / f"{measures[0]}.parquet").exists() else "sharded"
    n_shards = len(list((d / measures[0]).glob("*.parquet"))) if layout == "sharded" else 1
    return pd.Series(
        {"label": label, "path": str(d), "layout": layout,
         "n_measures": len(measures), "shards_per_measure": n_shards},
    )

## 1. What are we comparing?

In [ ]:
info = pd.DataFrame([describe_run(RUN_A, LABEL_A), describe_run(RUN_B, LABEL_B)])
display(info.set_index("label").T)

meas_a, meas_b = list_measures(RUN_A), list_measures(RUN_B)
aliased = {MEASURE_ALIASES.get(m, m) for m in meas_a}

print(f"measures in both        : {len(aliased & meas_b)}")
only_a = sorted(m for m in meas_a if MEASURE_ALIASES.get(m, m) not in meas_b)
only_b = sorted(meas_b - aliased)
if only_a:
    print(f"only in {LABEL_A}: {only_a}")
if only_b:
    print(f"only in {LABEL_B}: {only_b}")
if not only_a and not only_b:
    print("measure inventories match exactly")
if MEASURE_ALIASES:
    print(f"\napplied renames: {MEASURE_ALIASES}")

## 2. Comparison helpers

`compare_measure` is the workhorse. It reduces each run to **one total per random seed**,
then applies Welch's t-test to those two samples. Per-seed totals are the right unit: seeds
are the replication mechanism, so their spread *is* the run's noise floor.

Verdicts:

| verdict | meaning |
|---|---|
| `IDENTICAL` | every per-seed total matches exactly |
| `OK` | shift is small relative to seed noise (`|t| < T_REVIEW`) |
| `REVIEW` | `T_REVIEW <= |t| < T_DIFFERS` — suggestive, worth a look |
| `DIFFERS` | `|t| >= T_DIFFERS` — the distribution moved |

`t` here is not proof of a bug; it is a measure of whether the shift is distinguishable
from noise given the number of seeds you ran. More seeds narrow the noise floor and will
promote real-but-small shifts from `OK` to `DIFFERS`.

In [ ]:
def _scenario_col(df: pd.DataFrame) -> str | None:
    if SCENARIO_COL:
        return SCENARIO_COL if SCENARIO_COL in df.columns else None
    for c in ("scenario", "maternal_scenario"):
        if c in df.columns:
            return c
    return None


def _per_seed(df: pd.DataFrame, scenario: str | None) -> tuple[pd.Series, str]:
    """Total per random seed, optionally within one scenario.

    Most measures carry a ``value`` column and are summed. Line-list style outputs (the
    maternal ``births`` table) have no ``value``; those are compared on row count, which is
    the meaningful quantity for them. The metric used is returned so it can be reported --
    a row count and a value sum must never be silently conflated.
    """
    col = _scenario_col(df)
    if scenario is not None and col is not None:
        df = df[df[col] == scenario]
    metric = "sum(value)" if "value" in df.columns else "row count"
    if "random_seed" not in df.columns:
        total = df["value"].sum() if "value" in df.columns else float(len(df))
        return pd.Series({-1: total}), metric
    g = df.groupby("random_seed")
    return (g["value"].sum() if "value" in df.columns else g.size().astype(float)), metric


def compare_measure(measure_a: str, measure_b: str | None = None,
                    scenario: str | None = "__default__") -> dict:
    """Compare one measure across the two runs. Returns a row dict."""
    measure_b = measure_b or MEASURE_ALIASES.get(measure_a, measure_a)
    scenario = SCENARIO if scenario == "__default__" else scenario
    row = {"measure": measure_a, "scenario": scenario or "(all)"}
    try:
        a, metric_a = _per_seed(load_measure(RUN_A, measure_a), scenario)
        b, metric_b = _per_seed(load_measure(RUN_B, measure_b), scenario)
    except Exception as exc:  # a single unreadable measure must not abort the sweep
        return {**row, "verdict": "N/A", "note": f"{type(exc).__name__}: {exc}"}

    row["metric"] = metric_a if metric_a == metric_b else f"{metric_a} vs {metric_b}!"
    if metric_a != metric_b:
        return {**row, "verdict": "N/A", "note": "incomparable metrics"}

    common = sorted(set(a.index) & set(b.index))
    if not common:
        return {**row, "verdict": "N/A", "note": "no shared random seeds"}
    a, b = a.loc[common], b.loc[common]

    row.update({
        "n_seeds": len(common),
        f"mean_A": a.mean(), f"sd_A": a.std(),
        f"mean_B": b.mean(), f"sd_B": b.std(),
        "abs_diff": b.mean() - a.mean(),
        "pct_diff": (b.mean() - a.mean()) / a.mean() * 100 if a.mean() else np.nan,
        "seeds_exact": int((a.values == b.values).sum()),
    })
    row["shift_in_sd"] = abs(row["abs_diff"]) / a.std() if a.std() else np.nan

    if np.allclose(a.values, b.values, rtol=0, atol=0):
        row.update({"t": 0.0, "p": 1.0, "verdict": "IDENTICAL"})
        return row

    if _scipy_stats is not None and len(common) > 1:
        t, p = _scipy_stats.ttest_ind(b.values, a.values, equal_var=False)
    else:  # Welch by hand
        va, vb, n = a.var(ddof=1), b.var(ddof=1), len(common)
        se = np.sqrt(va / n + vb / n)
        t = (b.mean() - a.mean()) / se if se else np.nan
        p = np.nan
    row["t"], row["p"] = t, p
    at = abs(t)
    row["verdict"] = ("DIFFERS" if at >= T_DIFFERS
                      else "REVIEW" if at >= T_REVIEW else "OK")
    return row


def compare_all(measures=None, scenario="__default__") -> pd.DataFrame:
    """Compare every measure the two runs share."""
    if measures is None:
        measures = sorted(m for m in list_measures(RUN_A)
                          if MEASURE_ALIASES.get(m, m) in list_measures(RUN_B))
    rows = [compare_measure(m, scenario=scenario) for m in measures]
    df = pd.DataFrame(rows)
    order = {"DIFFERS": 0, "REVIEW": 1, "OK": 2, "IDENTICAL": 3, "N/A": 4}
    return df.sort_values("verdict", key=lambda s: s.map(order)).reset_index(drop=True)


def style_summary(df: pd.DataFrame):
    """Colour the verdict column. Text label always present -- never colour alone."""
    cols = [c for c in ("measure", "scenario", "metric", "n_seeds", "mean_A", "mean_B",
                        "pct_diff", "shift_in_sd", "t", "seeds_exact", "verdict")
            if c in df.columns]
    out = df[cols].rename(columns={"mean_A": f"mean [{LABEL_A}]", "mean_B": f"mean [{LABEL_B}]"})
    return (out.style
            .format({c: "{:,.2f}" for c in out.columns if out[c].dtype.kind == "f"})
            .apply(lambda s: [f"color: {STATUS_COLORS.get(v, INK)}; font-weight: 600"
                              for v in s] if s.name == "verdict" else ["" for _ in s]))

## 3. Headline: does everything agree?

One row per measure, worst verdict first. If this table is all `IDENTICAL` / `OK`, the two
runs agree to within their own noise and there is nothing to explain.

In [ ]:
summary = compare_all()
n_bad = (summary.verdict.isin(["DIFFERS", "REVIEW"])).sum()
print(f"{LABEL_A}  vs  {LABEL_B}   |   scenario: {SCENARIO or '(all)'}")
print(f"{len(summary)} measures compared -- "
      f"{(summary.verdict == 'IDENTICAL').sum()} identical, "
      f"{(summary.verdict == 'OK').sum()} within noise, "
      f"{(summary.verdict == 'REVIEW').sum()} to review, "
      f"{(summary.verdict == 'DIFFERS').sum()} differing")
style_summary(summary)

### How big is each shift, relative to that measure's own noise?

`shift_in_sd` = |mean difference| / per-seed SD of run A. Below ~1 the shift is comparable
to the scatter you would see between two seeds of the *same* run; well above it, the
distribution has genuinely moved. One axis, sorted by magnitude — the reference line marks
1 SD.

In [ ]:
def plot_shift(summary: pd.DataFrame, ax=None):
    d = summary.dropna(subset=["shift_in_sd"]).sort_values("shift_in_sd")
    if d.empty:
        print("nothing to plot")
        return
    if ax is None:
        _, ax = plt.subplots(figsize=(7.5, 0.36 * len(d) + 1.4))
    colors = [STATUS_COLORS.get(v, INK_MUTED) for v in d.verdict]
    y = np.arange(len(d))
    ax.barh(y, d.shift_in_sd, color=colors, height=0.62)
    ax.set_yticks(y, d.measure)
    ax.axvline(1.0, color=INK_MUTED, lw=1, ls="--", zorder=0)
    ax.text(1.0, len(d) - 0.35, " 1 SD", color=INK_MUTED, fontsize=9, va="top")
    ax.set_xlabel("|mean difference| / per-seed SD")
    ax.set_title(f"Shift relative to seed noise — {LABEL_B} vs {LABEL_A}", loc="left")
    ax.grid(axis="y", visible=False)
    # Direct-label each bar with the signed % change: identity never rests on colour.
    for yi, (v, pct) in enumerate(zip(d.shift_in_sd, d.pct_diff)):
        ax.text(v, yi, f"  {pct:+.2f}%", va="center", fontsize=9, color=INK_MUTED)
    ax.set_xlim(0, max(d.shift_in_sd.max() * 1.35, 1.25))
    handles = [plt.Line2D([], [], marker="s", ls="", markersize=8, color=STATUS_COLORS[k],
                          label=k) for k in ("IDENTICAL", "OK", "REVIEW", "DIFFERS")
               if k in set(d.verdict)]
    ax.legend(handles=handles, loc="lower right", fontsize=9)
    plt.tight_layout()
    return ax


plot_shift(summary)
plt.show()

## 4. Drill into one measure

Per-seed totals for both runs side by side. This is the plot that settles "is the
difference real or is it noise" by eye: if the two clouds interleave, the runs agree.

In [ ]:
def plot_seeds(measure_a: str, measure_b: str | None = None,
               scenario: str = "__default__", ax=None):
    measure_b = measure_b or MEASURE_ALIASES.get(measure_a, measure_a)
    scenario = SCENARIO if scenario == "__default__" else scenario
    a, _ = _per_seed(load_measure(RUN_A, measure_a), scenario)
    b, _ = _per_seed(load_measure(RUN_B, measure_b), scenario)
    common = sorted(set(a.index) & set(b.index))
    a, b = a.loc[common], b.loc[common]

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 3.6))
    x = np.arange(len(common))
    ax.plot(x, a.values, "o", color=COLOR_A, markersize=8, label=LABEL_A)
    ax.plot(x, b.values, "o", color=COLOR_B, markersize=8, label=LABEL_B)
    for xi, (av, bv) in enumerate(zip(a.values, b.values)):
        ax.plot([xi, xi], [av, bv], color="#d8d6d0", lw=1.5, zorder=0)
    ax.axhline(a.mean(), color=COLOR_A, lw=1.5, ls="--", alpha=0.7)
    ax.axhline(b.mean(), color=COLOR_B, lw=1.5, ls="--", alpha=0.7)
    ax.set_xticks(x, [str(s) for s in common])
    ax.set_xlabel("random seed")
    ax.set_ylabel(measure_a)
    ax.set_title(f"{measure_a} per seed — scenario: {scenario or '(all)'}", loc="left")
    ax.legend(loc="best", fontsize=9)
    ax.grid(axis="x", visible=False)
    plt.tight_layout()
    return ax


worst = summary.iloc[0]["measure"] if len(summary) else None
if worst:
    plot_seeds(worst)
    plt.show()

## 5. Where inside a measure does the difference sit?

Aggregate agreement can hide offsetting shifts. This breaks one measure down by any
stratification column and reports the difference per stratum.

In [ ]:
def compare_strata(measure_a: str, by: str | list[str],
                   measure_b: str | None = None, scenario: str = "__default__",
                   top: int = 25) -> pd.DataFrame:
    """Difference per stratum, largest relative gap first."""
    measure_b = measure_b or MEASURE_ALIASES.get(measure_a, measure_a)
    scenario = SCENARIO if scenario == "__default__" else scenario
    by = [by] if isinstance(by, str) else list(by)

    def agg(run, measure):
        df = load_measure(run, measure)
        col = _scenario_col(df)
        if scenario is not None and col is not None:
            df = df[df[col] == scenario]
        missing = [c for c in by if c not in df.columns]
        if missing:
            raise KeyError(f"{missing} not in {measure} ({sorted(df.columns)})")
        g = df.groupby(by, observed=True)
        return g["value"].sum() if "value" in df.columns else g.size().astype(float)

    a, b = agg(RUN_A, measure_a), agg(RUN_B, measure_b)
    out = pd.DataFrame({LABEL_A: a, LABEL_B: b}).fillna(0.0)
    out["abs_diff"] = out[LABEL_B] - out[LABEL_A]
    out["pct_diff"] = np.where(out[LABEL_A] != 0,
                               out["abs_diff"] / out[LABEL_A] * 100, np.nan)
    return out.reindex(out.pct_diff.abs().sort_values(ascending=False).index).head(top)


# Scenario is the most common place for a difference to hide -- compare across all of them.
if worst:
    df0 = load_measure(RUN_A, worst)
    scol = _scenario_col(df0)
    if scol:
        display(compare_strata(worst, scol, scenario=None))
    else:
        print(f"{worst} has no scenario column")

In [ ]:
# Any stratification column works. Uncomment / edit as needed:
# compare_strata(worst, "age_group")
# compare_strata(worst, "wealth_quintile")
# compare_strata(worst, ["age_group", "wealth_quintile"], top=15)

## 6. Add your own comparisons below

Everything above is built from four functions, all safe to call ad hoc:

```python
load_measure(RUN_A, "deaths")                 # tidy frame, either layout
compare_measure("ylds")                       # one verdict row
compare_all(["deaths", "ylls"])               # summary table
compare_strata("deaths", "age_group")         # per-stratum breakdown
plot_seeds("deaths"); plot_shift(summary)     # the two plots
```

To compare a different pair of runs, edit the parameters cell and re-run — or reassign
`RUN_A` / `RUN_B` in a cell here to compare several pairs in one pass. To compare the child
sim instead of maternal, point both paths at the `child/...` directories; `SCENARIO_COL`
auto-detects `maternal_scenario`.

In [ ]:
# Scratch cell -- e.g. sweep every vehicle/location for one measure.
#
# NEW_ROOT = "/mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model1.0"
# OLD_ROOT = ("/mnt/team/simulation_science/pub/models/vivarium_gates_lsff_by_wealth_quintile"
#             "/2026_08_13_13_55_48")
# rows = []
# for sim, old_sim in [("maternal", "0200_pregnancy_sim"), ("child", "0300_child_sim")]:
#     for vehicle, location in [("rice", "nigeria"), ("rice", "india"), ("bouillon", "nigeria")]:
#         RUN_A = f"{OLD_ROOT}/{old_sim}/sim_results/{vehicle}/{location}"
#         RUN_B = f"{NEW_ROOT}/{sim}/{vehicle}/{location}"
#         r = compare_measure("deaths")
#         rows.append({"sim": sim, "combo": f"{vehicle}/{location}", **r})
# pd.DataFrame(rows)